# 11 - Cluster Profiling and Default Analysis
### Credit Portfolio Segmentation and Risk Pattern Analysis

---

## Validating Segments Against Historical Default Rates
Here is where the business picture emerges:
1. We bring back the ground-truth **`Default`** column.
2. We examine whether our unsupervised behavioral clusters naturally separated safe borrowers from risky borrowers.
3. We assign descriptive **Business Personas** to each segment.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

df = pd.read_csv("../data/processed/cleaned_loans_dataset.csv")
X_scaled = pd.read_csv("../data/processed/scaled_features.csv")

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['KMeans_Cluster'] = kmeans.fit_predict(X_scaled)

print("Clusters fitted and assigned.")


---
## 1. Calculating Cluster Averages & Default Rates


In [ ]:
profile = df.groupby('KMeans_Cluster').agg(
    Borrower_Count=('Default', 'count'),
    Avg_Income=('annual_income', 'mean'),
    Avg_FICO=('fico_score', 'mean'),
    Avg_DTI=('dti', 'mean'),
    Avg_Loan_Amount=('loan_amount', 'mean'),
    Default_Rate_Pct=('Default', lambda x: round(x.mean() * 100, 2))
).reset_index()

profile.round(2)


---
## 2. Naming Borrower Personas
Based on financial characteristics and default behavior, we give each segment a clear business identity:


In [ ]:
persona_map = {
    0: "Prime / Low Risk",
    1: "Overleveraged / Moderate Risk",
    2: "Entry-Level / Young Profile",
    3: "Subprime / High Risk"
}

df['Segment_Name'] = df['KMeans_Cluster'].map(persona_map)
profile['Segment_Name'] = profile['KMeans_Cluster'].map(persona_map)

profile.to_csv("../outputs/tables/cluster_risk_profiles.csv", index=False)
profile[['Segment_Name', 'Borrower_Count', 'Avg_Income', 'Avg_FICO', 'Default_Rate_Pct']]


---
## 3. Visualizing Default Rates Across Segments


In [ ]:
plt.figure(figsize=(9, 5))
sns.barplot(x='Segment_Name', y='Default_Rate_Pct', data=profile, palette='magma')
plt.title("Historical Loan Default Rate by Borrower Segment (%)")
plt.xlabel("Borrower Segment")
plt.ylabel("Default Rate (%)")
plt.xticks(rotation=15)

for idx, row in profile.iterrows():
    plt.text(idx, row['Default_Rate_Pct'] + 0.5, f"{row['Default_Rate_Pct']}%", ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig("../outputs/figures/cluster_default_rates.png", dpi=300)
plt.show()

df.to_csv("../data/processed/segmented_loans_dataset.csv", index=False)
print("Segmented portfolio saved to ../data/processed/segmented_loans_dataset.csv")
